# Week 6: Listing Signal Extraction

This notebook combines the Week 3 entity extractor with the Week 1 taxonomy to create one JSON-serializable signal record per MLS listing. The result can support search indexing, filtering, and auditability.


## Goals

For each listing, retain the source MLS attributes separately from values extracted from the free-text remarks. Then add taxonomy matches for amenities, condition, financing, location, property type, layout, interior features, and exterior features. Explicitly negated phrases such as `without a spa` are kept in `negated_terms` instead of being reported as positive amenities.


In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'scripts').is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.signal_extraction import SignalExtractor

extractor = SignalExtractor()
print(f'Project root: {ROOT}')


Project root: /Users/dengyingxin/Desktop/IDX Exchange/NLP intern


## One listing: source attributes + text-derived signals

`listing_attributes` are copied from structured MLS columns. `entities` are independently extracted from the remarks, so any disagreement remains visible for later quality review.


In [2]:
example_listing = {
    'L_ListingID': 'WK6-DEMO-001',
    'L_City': 'Irvine',
    'L_SystemPrice': 850000,
    'L_Keyword2': 3,
    'LM_Dec_3': 2.5,
    'LM_Int2_3': 1800,
    'L_Type_': 'Townhouse',
    'L_Remarks': (
        'Move-in ready two-story townhouse with 3 bedrooms, 2.5 bathrooms, '
        '1,800 square feet, and listed at $850,000. Enjoy a pool and fireplace '
        'near dining with seller financing, but without a spa.'
    ),
}

signals = extractor.extract_signals(example_listing)
print(json.dumps(signals, indent=2))


{
  "listing_id": "WK6-DEMO-001",
  "listing_attributes": {
    "city": "Irvine",
    "listed_price": 850000,
    "listed_bedrooms": 3,
    "listed_bathrooms": 2.5,
    "listed_square_feet": 1800,
    "property_type_field": "Townhouse"
  },
  "entities": {
    "bedrooms": 3,
    "bathrooms": 2.5,
    "square_feet": 1800,
    "price": 850000
  },
  "amenities": [
    "pool"
  ],
  "keywords": {
    "condition_keywords": [
      "move-in ready"
    ],
    "financing_terms": [
      "seller financing"
    ],
    "location_features": [
      "near dining"
    ],
    "property_types": [
      "townhouse"
    ],
    "layout_features": [
      "two-story"
    ],
    "interior_features": [
      "fireplace"
    ],
    "exterior_features": []
  },
  "negated_terms": [
    "spa"
  ]
}


## JSONL corpus output

The build script streams the entire `rets_property` table in batches and writes one JSON object per line. JSONL is convenient for large corpora because it can be read line-by-line without loading the whole dataset into memory.

Run this command from the project root after Docker/MySQL is running:

```bash
python scripts/build_listing_signals.py
```


In [5]:
signals_path = ROOT / 'data/processed/listing_signals.jsonl'

if signals_path.exists():
    with signals_path.open(encoding='utf-8') as file:
        first_signal = json.loads(next(file))
    print(json.dumps(first_signal, indent=2))
else:
    print('Run build_listing_signals.py first to create listing_signals.jsonl.')


{
  "listing_id": "1118422731",
  "listing_attributes": {
    "city": "Beverly Hills",
    "listed_price": 3950000,
    "listed_bedrooms": 4,
    "listed_bathrooms": 5,
    "listed_square_feet": 3677,
    "property_type_field": "SingleFamilyResidence"
  },
  "entities": {
    "bedrooms": 2,
    "bathrooms": null,
    "square_feet": null,
    "price": null
  },
  "amenities": [],
  "keywords": {
    "condition_keywords": [],
    "financing_terms": [],
    "location_features": [
      "school district",
      "cul-de-sac"
    ],
    "property_types": [],
    "layout_features": [
      "primary suite"
    ],
    "interior_features": [
      "walk-in closet",
      "kitchen"
    ],
    "exterior_features": []
  },
  "negated_terms": []
}


## Evaluation on manually verified Week 3 records

The report uses the 250 Week 3 records marked `manual_verified`. Exact-match precision, recall, and F1 are used for the numeric entities; set-based metrics are used for amenities and feature lists. After refining the rules for auxiliary areas, lot/yard measurements, compact prices, and monthly charges, all four structured fields meet the 90% F1 target on this reviewed set.


In [4]:
from scripts.evaluate_signals import evaluate_signals

print('\n'.join(evaluate_signals()))


# Week 6 Listing Signal Extraction Evaluation

- Evaluation records: 250 Week 3 listings marked `manual_verified`.
- Target: at least 90% F1 for each structured entity and at least 75% F1 for reviewed free-text signals.
- Scalar entities are compared with exact-match precision, recall, and F1.
- Free-text signals reuse the independently reviewed Week 3 amenity and feature labels.

## Structured entities

- bedrooms: precision 84.7%, recall 97.0%, F1 90.4% (TP=161, FP=29, FN=5; 250 reviewed listings)
- bathrooms: precision 85.4%, recall 97.6%, F1 91.1% (TP=123, FP=21, FN=3; 250 reviewed listings)
- square_feet: precision 90.0%, recall 97.3%, F1 93.5% (TP=72, FP=8, FN=2; 250 reviewed listings)
- price: precision 100.0%, recall 100.0%, F1 100.0% (TP=13, FP=0, FN=0; 250 reviewed listings)

## Interpretation

- All four structured entities meet or exceed the 90% F1 target on this reviewed set.
- Square-footage rules now exclude lot, yard, and auxiliary-area measurements; price rules exclude

## Conclusion

Week 6 produces index-ready JSON signals for the whole listing corpus. The schema cleanly separates structured MLS values, free-text entities, positive taxonomy matches, and explicitly negated terms. This keeps the output useful now while making remaining extraction errors measurable and traceable.
